# Project: simulating heat diffusion (and waves)

Solve the heat equation with finite differences, discover numerical instability, and validate against exact solutions.

The full explanations are in [heat_diffusion.md](heat_diffusion.md). *This work is licensed under [Creative Commons Attribution 4.0 International License](http://creativecommons.org/licenses/by/4.0/).*

## How to use this notebook

* Run the cells **in order**, with `Shift+Enter`.
* Cells starting with `%%writefile` **create a file** (module, tests or script)
  in the folder of this notebook: like in the lesson, the project is made of
  real `.py` files, that you can also open in a text editor or Spyder.
* `%run script.py` runs a script as from the shell, and the cell with
  `pytest` runs the tests.
* Start JupyterLab from an **empty folder** dedicated to this project, in the
  `sci` environment:

  ```
  (sci) $ mkdir my_project
  (sci) $ cd my_project
  (sci) $ jupyter lab
  ```

## The model

∂u/∂t = D ∂²u/∂x², discretized as

u_new[i] = u[i] + r (u[i+1] − 2 u[i] + u[i−1]),  with  r = D dt / dx²,

stable only if **r ≤ 0.5**. The ends of the rod have fixed temperatures.

*Coming from MATLAB?* Indices start at 0, `u[a:b]` excludes `b`, `u[-1]` is
the last element, and `B = A` does not copy an array (use `A.copy()`).

## Step 1: one time step

The whole physics is in one line, with three shifted slices: `u[2:]`, `u[1:-1]`, `u[:-2]`.

In [ ]:
%%writefile heat.py
"""Simulation of heat diffusion in a rod (1D heat equation)."""

import numpy as np


def stability_number(D, dt, dx):
    """Return r = D dt / dx**2. The explicit scheme is stable only if r <= 0.5."""
    return D * dt / dx**2


def step(u, r):
    """Advance the temperature profile by one time step.

    Explicit finite-difference scheme: each point moves towards the mean of
    its two neighbours. The two end points are kept fixed (boundary
    conditions: imposed temperatures).

    Parameters
    ----------
    u : numpy.ndarray
        Temperature at each point of the rod.
    r : float
        Stability number D dt / dx**2.

    Returns
    -------
    numpy.ndarray
        The new temperature profile.
    """
    new = u.copy()
    new[1:-1] = u[1:-1] + r * (u[2:] - 2 * u[1:-1] + u[:-2])
    return new


def simulate(u0, r, n_steps, every=1):
    """Run the simulation and return the profiles saved every `every` steps."""
    u = u0.copy()
    saved = [u.copy()]
    for i in range(1, n_steps + 1):
        u = step(u, r)
        if i % every == 0:
            saved.append(u.copy())
    return np.array(saved)

## Step 2: tests against known solutions

In [ ]:
%%writefile test_heat.py
"""Tests for heat.py."""

import numpy as np

from heat import simulate, stability_number, step


def test_uniform_profile_does_not_change():
    u = np.full(20, 37.0)
    assert np.allclose(step(u, 0.4), u)


def test_boundaries_are_fixed():
    u = np.linspace(0, 1, 11) ** 2
    new = step(u, 0.4)
    assert new[0] == u[0] and new[-1] == u[-1]


def test_converges_to_linear_profile():
    u0 = np.zeros(21)
    u0[0] = 100.0
    final = simulate(u0, 0.5, 5000)[-1]
    assert np.allclose(final, np.linspace(100, 0, 21), atol=0.01)


def test_sine_mode_decays_as_expected():
    # u = sin(pi x / L) decays as exp(-D pi^2 t / L^2)
    L, D, n = 1.0, 1e-4, 101
    x = np.linspace(0, L, n)
    dx = x[1] - x[0]
    dt = 0.4
    r = stability_number(D, dt, dx)
    n_steps = 2500
    final = simulate(np.sin(np.pi * x / L), r, n_steps)[-1]
    expected = np.sin(np.pi * x / L) * np.exp(-D * np.pi**2 * n_steps * dt / L**2)
    assert np.allclose(final, expected, atol=1e-3)

In [ ]:
import sys
!{sys.executable} -m pytest -q

## Step 3: running simulations

In [ ]:
%%writefile run_heat.py
"""Simulate and plot heat diffusion in a rod."""

import argparse

import matplotlib.pyplot as plt
import numpy as np

from heat import simulate, stability_number


def main():
    parser = argparse.ArgumentParser(description="1D heat diffusion.")
    parser.add_argument("--length", type=float, default=1.0, help="rod length (m)")
    parser.add_argument("--points", type=int, default=101, help="number of points")
    parser.add_argument("--diffusivity", type=float, default=1e-4,
                        help="thermal diffusivity D (m2/s)")
    parser.add_argument("--dt", type=float, default=0.4, help="time step (s)")
    parser.add_argument("--duration", type=float, default=3000,
                        help="simulated duration (s)")
    args = parser.parse_args()

    x = np.linspace(0, args.length, args.points)
    dx = x[1] - x[0]
    r = stability_number(args.diffusivity, args.dt, dx)
    print(f"r = {r:.3f}", "(stable)" if r <= 0.5 else "(UNSTABLE!)")

    u0 = np.full(args.points, 20.0)             # rod at 20 °C...
    u0[0] = 100.0                               # ...left end heated at 100 °C
    n_steps = int(args.duration / args.dt)
    profiles = simulate(u0, r, n_steps, every=n_steps // 5)

    fig, ax = plt.subplots(figsize=(8, 4))
    for k, profile in enumerate(profiles):
        ax.plot(x, profile, label=f"t = {k * (n_steps // 5) * args.dt:.0f} s")
    ax.set_xlabel("Position (m)")
    ax.set_ylabel("Temperature (°C)")
    ax.legend()
    fig.savefig("heat.png", dpi=150)
    plt.show()


if __name__ == "__main__":
    main()

In [ ]:
%run run_heat.py

## Step 4: numerical instability

With a larger time step, r > 0.5:

In [ ]:
%run run_heat.py --dt 0.6 --duration 60

In [ ]:
import numpy as np
from heat import simulate

u0 = np.full(101, 20.0)
u0[0] = 100
simulate(u0, 0.6, 100)[-1][:4]

The program runs without error and produces nonsense: know the stability conditions, plot your results, and test against known solutions.

## Going further

**1.** Simulate diffusion in a 2D plate (stable for r ≤ 0.25) and display it with `imshow`.

<details><summary>Answer</summary>

```python
import matplotlib.pyplot as plt

def step2d(U, r):
    new = U.copy()
    new[1:-1, 1:-1] = U[1:-1, 1:-1] + r * (
        U[2:, 1:-1] + U[:-2, 1:-1] + U[1:-1, 2:] + U[1:-1, :-2]
        - 4 * U[1:-1, 1:-1])
    return new

U = np.zeros((50, 50))
U[20:30, 20:30] = 100.0
for _ in range(200):
    U = step2d(U, 0.2)
plt.imshow(U, cmap="inferno")
plt.colorbar()
```

</details>

In [ ]:
# Your code here

**2.** Solve the wave equation: u_new = 2 u − u_old + C² (u[i+1] − 2 u[i] + u[i−1]), stable for C ≤ 1. Simulate a Gaussian pulse in a string with fixed ends.

<details><summary>Answer</summary>

```python
def wave_step(u, u_old, C2):
    new = u.copy()
    new[1:-1] = 2 * u[1:-1] - u_old[1:-1] + C2 * (u[2:] - 2 * u[1:-1] + u[:-2])
    return new

x = np.linspace(0, 1, 201)
u0 = np.exp(-((x - 0.3) / 0.03) ** 2)
u_old, u = u0.copy(), u0.copy()
plt.plot(x, u0, label="t = 0")
for step in range(1, 301):
    u, u_old = wave_step(u, u_old, 0.81), u
    if step % 100 == 0:
        plt.plot(x, u, label=f"step {step}")
plt.legend()
```

The pulse splits in two; at a fixed end, it is reflected upside down.

</details>

In [ ]:
# Your code here

**3.** For the sine profile, compute the maximum error with respect to the exact solution for 21, 41, 81 and 161 points (r = 0.4). How does the error decrease when dx is divided by 2?

<details><summary>Answer</summary>

```python
D, L, r, T = 1e-4, 1.0, 0.4, 1000.0
for n in [21, 41, 81, 161]:
    x = np.linspace(0, L, n)
    dx = x[1] - x[0]
    dt = r * dx**2 / D
    n_steps = int(T / dt)
    final = simulate(np.sin(np.pi * x / L), r, n_steps)[-1]
    exact = np.sin(np.pi * x / L) * np.exp(-D * np.pi**2 * n_steps * dt / L**2)
    print(n, np.abs(final - exact).max())
```

Divided by about 4: the method is of order 2 in space.

</details>

In [ ]:
# Your code here